### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.


In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response


AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds known for mimicking human speech, right? But why do they do that? Maybe it\'s related to their natural behavior. I remember that some birds use mimicry to communicate. Maybe parrots evolved this ability for some reason.\n\nFirst, I should think about their habitat. Parrots live in environments with a lot of other bird species. Maybe they mimic other birds\' calls to blend in or to communicate with their own kind. But then why would they mimic humans specifically? That\'s when humans come into play. When kept as pets, parrots might pick up words from their surroundings. But does that mean they understand the words, or are they just imitating sounds?\n\nAnother angle is social behavior. Parrots are social animals. In the wild, they might mimic sounds to establish social bonds, attract mates, or warn others. If they replicate human speech, maybe i

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'vdjf4jqnj', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.164779352, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006046432, 'prompt_tokens_details': None, 'queue_time': 0.056344478, 'total_time': 0.170825784}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019b6e7f-34df-70b0-b

### Tool Execution Loop

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': '9djy748gd', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 153, 'total_tokens': 244, 'completion_time': 0.135568732, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.009184728, 'prompt_tokens_details': None, 'queue_time': 0.282985831, 'total_time': 0.14475346}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_rea